# Vòng 61: huấn luyện backbone rồi trích đặc trưng

Mỗi run: `cure-ood train` (ghi luôn cấu hình trích và benchmark) ⭢ `cure-ood extract` ⭢ `--select-k`.
Train tự chạy tiếp từ `state.pt`, nên hết phiên 12 giờ thì gắn Output lần trước làm Input và chạy lại.
Đầu ra: `/kaggle/working/r61_out.zip` (checkpoint, đặc trưng, cấu hình, log).

In [ ]:
RUNS = ["r61_resnet20", "r61_mobilenetv2", "r61_resnet18embed128", "r61_resnet18_s0"]  # chia theo phiên
EPOCHS = None     # None = theo cấu hình (100); đặt số nhỏ để thử nhanh
SUBSAMPLE = None  # None = toàn bộ; đặt vd 2000 để thử nhanh

In [ ]:
print("@@CELL 1@@", flush=True)
import json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path
INPUT, WORK = Path("/kaggle/input"), Path("/kaggle/working")
T0 = time.time()
def tick(m): print(f"[{(time.time() - T0) / 60:6.1f} phút] {m}", flush=True)

code_dir = next(t.parent for t in sorted(INPUT.rglob("pyproject.toml")) if 'name = "cure-ood"' in t.read_text(encoding="utf-8"))
REPO = Path("/tmp/CURE")  # ngoài /kaggle/working: output chỉ gồm thứ ta chủ động nén
if not REPO.exists():
    shutil.copytree(code_dir, REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(REPO)], check=True)
sys.path.insert(0, str(REPO / "src"))

# Kaggle giải nén tar vào thư mục trùng tên nên có thể lồng hai lớp: chọn thư mục thật sự chứa cifar10.
IMG = next(p for p in INPUT.rglob("images_classic") if (p / "cifar10").is_dir())
LIST = next(p for p in INPUT.rglob("benchmark_imglist") if (p / "cifar10").is_dir())
FEAT = WORK / "features"; FEAT.mkdir(exist_ok=True)
os.environ.update(CURE_IMAGES=str(IMG.parent), CURE_IMGLISTS=str(LIST), CURE_FEATURES=str(FEAT), CURE_HF_AUTO="0")
os.chdir(REPO)
print(open("VERSION.txt").read())
print("ảnh:", sorted(p.name for p in IMG.iterdir())[:10])

# Nạp lại kết quả phiên trước (checkpoint dở, đặc trưng đã trích).
prev = next((p for p in INPUT.rglob("r61_out.zip")), None)
if prev:
    with zipfile.ZipFile(prev) as z:
        z.extractall(REPO)
    for f in (REPO / "features").glob("*.npz") if (REPO / "features").exists() else []:
        if not (FEAT / f.name).exists():
            shutil.copy(f, FEAT / f.name)
    tick(f"nạp lại phiên trước: {sorted(p.name for p in (REPO / 'results/train').iterdir())}" if (REPO / "results/train").exists() else "nạp lại")
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ")
assert torch.cuda.is_available(), "cần GPU"

In [ ]:
print("@@CELL 2@@", flush=True)
EXTRA = []
if EPOCHS:
    EXTRA += ["--set", f"stages.0.epochs={EPOCHS}"]
if SUBSAMPLE:
    EXTRA += ["--set", f"data.subsample={SUBSAMPLE}"]

def sh(cmd, tag):
    """Chạy một lệnh, ghi log ra file, trả về mã thoát."""
    log = REPO / "results/logs" / f"{tag}.log"
    log.parent.mkdir(parents=True, exist_ok=True)
    with open(log, "a", encoding="utf-8") as f:
        p = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT)
    return p.returncode

done, failed = [], []
for run in RUNS:
    cfg = f"configs/train/{run}.yaml"
    if sh([sys.executable, "-m", "cure_ood.cli", "train", cfg, *EXTRA], f"train_{run}"):
        failed.append((run, "train")); tick(f"LỖI train {run}"); continue
    tick(f"train xong {run}")
    if sh([sys.executable, "-m", "cure_ood.cli", "extract", run], f"extract_{run}"):
        failed.append((run, "extract")); tick(f"LỖI extract {run}"); continue
    if sh([sys.executable, "-m", "cure_ood.cli", "train", cfg, *EXTRA, "--select-k"], f"selectk_{run}"):
        failed.append((run, "select-k")); tick(f"LỖI select-k {run}"); continue
    done.append(run); tick(f"xong hẳn {run}")
print("xong:", done, "| lỗi:", failed)

In [ ]:
print("@@CELL 3@@", flush=True)
for run in done:
    b = REPO / f"configs/benchmarks/{run}.yaml"
    log = (REPO / "results/train" / run / "log.jsonl")
    last = json.loads(log.read_text(encoding="utf-8").splitlines()[-1]) if log.exists() else {}
    print(f"{run:24s} k={[l for l in b.read_text(encoding='utf-8').splitlines() if l.startswith('k:')]} "
          f"val_acc={last.get('val_acc')} epoch={last.get('epoch')}", flush=True)

out = WORK / "r61_out.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for pat in ("results/train/**/*", "results/logs/*", "configs/extract/r61_*", "configs/benchmarks/r61_*"):
        for f in REPO.glob(pat):
            if f.is_file():
                z.write(f, f.relative_to(REPO))
    for f in FEAT.glob("r61_*.npz"):
        z.write(f, Path("features") / f.name)
print(out, round(out.stat().st_size / 1e6, 1), "MB")